In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import random

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Class labels
class_columns = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']

# Load CSVs
train_df = pd.read_csv("train_df_processed.csv")
val_df = pd.read_csv("val_df_processed.csv")

# Dataset and transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx]['image_path']
        label = self.data.iloc[idx]['label_idx']
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

train_ds = SkinLesionDataset(train_df, transform=train_transform)
val_ds = SkinLesionDataset(val_df, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)

# Model setup
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT  # Use the pretrained ImageNet weights
model = resnet18(weights=weights)


# Unfreeze last two layers (layer4 + fc) for fine-tuning, freeze rest
for name, param in model.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(class_columns))
model = model.to(device)

# Class weights to handle imbalance
class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label_idx']),
    y=train_df['label_idx']
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Training loop with validation and best model saving
def train_model(model, criterion, optimizer, train_loader, val_loader, scheduler=None, epochs=5):
    best_val_acc = 0.0
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)

        for images, labels in train_bar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            train_bar.set_postfix(loss=loss.item(), accuracy=correct/total)

        train_acc = correct / total
        avg_loss = total_loss / len(train_loader)

        if scheduler:
            scheduler.step()

        val_acc = evaluate_model(model, val_loader, class_columns, print_report=False)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_resnet18_model.pth")
            print(f"Best model saved with val acc: {best_val_acc:.4f}")

    print(f"Training complete. Best val acc: {best_val_acc:.4f}")

# Evaluation function
def evaluate_model(model, val_loader, class_names, print_report=True):
    model.eval()
    val_correct = 0
    val_total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc="Validation", leave=False)
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            val_bar.set_postfix(val_accuracy=val_correct/val_total)

    val_acc = val_correct / val_total
    if print_report:
        print(f"Validation Accuracy: {val_acc:.4f}")
        print("\nClassification Report:")
        print(classification_report(all_labels, all_preds, target_names=class_names))
    return val_acc

# Example usage: train for 10 epochs
train_model(model, criterion, optimizer, train_loader, val_loader, scheduler, epochs=5)

# Load best model weights before final evaluation (optional)
model.load_state_dict(torch.load("best_resnet18_model.pth"))
evaluate_model(model, val_loader, class_columns)


Using device: cpu


Epoch 1/5 [Train]:   9%|▉         | 47/501 [00:23<03:35,  2.11it/s, accuracy=0.327, loss=1.62] 

In [1]:
# --------------------
# Imports and Setup
# --------------------
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------
# Load CSV files
# --------------------
train_df = pd.read_csv("train_df_processed.csv")
val_df = pd.read_csv("val_df_processed.csv")

# --------------------
# Label mapping
# --------------------
class_columns = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
class_to_idx = {c: i for i, c in enumerate(class_columns)}

# --------------------
# Image Transforms
# --------------------
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

# --------------------
# Dataset Class
# --------------------
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx]['image_path']
        label = self.data.iloc[idx]['label_idx']
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# --------------------
# Create DataLoaders
# --------------------
train_ds = SkinLesionDataset(train_df, transform=train_transform)
val_ds = SkinLesionDataset(val_df, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)


In [30]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import time
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [31]:
# Define class labels
class_columns = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
class_to_idx = {c: i for i, c in enumerate(class_columns)}

# Load training CSV
train_df = pd.read_csv("train_df_processed.csv")
val_df = pd.read_csv("val_df_processed.csv")


# hi

In [32]:
model = models.resnet18(pretrained=True)

# Freeze all pretrained layers
for param in model.parameters():
    param.requires_grad = False

# Replace and unfreeze only the final layer
num_features = model.fc.in_features
num_classes = 7  # ISIC 2018 has 7 classes
model.fc = nn.Linear(num_features, num_classes)

# Ensure the new final layer is trainable
for param in model.fc.parameters():
    param.requires_grad = True

# Move model to device
model = model.to(device)


/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [33]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label_idx']),
    y=train_df['label_idx']
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)


In [34]:
optimizer = optim.Adam(model.fc.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)


In [35]:
from torch.utils.data import Dataset
from PIL import Image

class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx]['image_path']
        label = self.data.iloc[idx]['label_idx']
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


In [36]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

from torch.utils.data import DataLoader

train_ds = SkinLesionDataset(train_df, transform=train_transform)
val_ds = SkinLesionDataset(val_df, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)



In [37]:
import time
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def train_model(model, criterion, optimizer, train_loader, scheduler=None, epochs=10):
    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        total_loss, correct, total = 0, 0, 0
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)

        for images, labels in train_bar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            train_bar.set_postfix(loss=loss.item(), accuracy=correct/total)

        train_acc = correct / total
        avg_loss = total_loss / len(train_loader)

        if scheduler:
            scheduler.step()

        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} | Time: {epoch_time:.1f}s")


In [38]:
def evaluate_model(model, val_loader, class_names):
    model.eval()
    val_correct, val_total = 0, 0
    all_preds = []
    all_labels = []
    val_bar = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            val_bar.set_postfix(val_accuracy=val_correct/val_total)

    val_acc = val_correct / val_total
    print(f"Validation Accuracy: {val_acc:.4f}")

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    return all_labels, all_preds


In [39]:
torch.save(model.state_dict(), "best_resnet18_model.pth")


In [ ]:
def evaluate_model(model, val_loader, class_names):
    model.eval()
    val_correct, val_total = 0, 0
    all_preds = []
    all_labels = []
    val_bar = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            val_bar.set_postfix(val_accuracy=val_correct/val_total)

    val_acc = val_correct / val_total
    print(f"Validation Accuracy: {val_acc:.4f}")

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    return all_labels, all_preds


In [40]:
# Train for one epoch (or more)
train_model(model, criterion, optimizer, train_loader, scheduler, epochs=1)

Epoch 1/1 | Train Loss: 1.8676 | Train Acc: 0.4678 | Time: 140.7s


In [41]:

# Evaluate separately whenever you want
evaluate_model(model, val_loader, class_to_idx)

Validation Accuracy: 0.4838

Classification Report:
              precision    recall  f1-score   support

         MEL       0.19      0.48      0.27       223
          NV       0.92      0.55      0.69      1341
         BCC       0.19      0.40      0.25       103
       AKIEC       0.05      0.06      0.05        65
         BKL       0.26      0.32      0.29       220
          DF       0.00      0.00      0.00        23
        VASC       0.15      0.25      0.18        28

    accuracy                           0.48      2003
   macro avg       0.25      0.29      0.25      2003
weighted avg       0.68      0.48      0.54      2003



([1,
  1,
  1,
  1,
  1,
  2,
  4,
  4,
  4,
  1,
  0,
  6,
  1,
  0,
  1,
  2,
  0,
  1,
  0,
  4,
  0,
  1,
  1,
  1,
  4,
  1,
  1,
  1,
  1,
  3,
  1,
  1,
  2,
  1,
  1,
  4,
  3,
  1,
  1,
  1,
  0,
  4,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  4,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  4,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  2,
  0,
  0,
  0,
  0,
  1,
  3,
  2,
  1,
  2,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  4,
  1,
  1,
  4,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  4,
  1,
  1,
  0,
  1,
  1,
  1,
  4,
  0,
  1,
  2,
  1,
  1,
  4,
  1,
  1,
  0,
  1,
  1,
  1,
  4,
  1,
  1,
  1,
  1,
  1,
  4,
  6,
  4,
  1,
  0,
  1,
  1,
  1,
  0,
  2,
  0,
  5,
  3,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  5,
  4,
  1,
  1,
  1,
  1,
  3,
  1,
  1,
  1,
  2,
  4,
  4,
  1,
  4,
  1,
  1,
  3,
  1,
  1,
  4,
  4,
  2,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  4,
  1,
  1,
  0,
  1,
  4,
  1,
  1,
  1,
  6,
  1,
  1,
  4,
